#### 1.原始訊息
##### 收到日期：20260821
##### 完成日期：2026
##### 花費時間：

- 再來想想… 
    - 像這樣程式交易的數據如何呈現，你有勇氣進場？進場後發生什麼事，會覺得這個策略有問題，需停用？ 
    - 這是個單純指數的程式交易，不用複雜產業、個股與籌碼資訊，已經能獲利了！您只花了1.5天就找到可以賺錢的策略，為何市場上大多數人都賠錢？
    - 同樣邏輯，增加一個商品「台指期（TX）」，加權指數觸發進出場時時，以TX當天收盤執行。統計以下結果，以表格呈現 
        - 總交易
        - 淨損益
        - 勝率
        - 平均賺賠比
        - 平均持有期
        - 最大回檔
        - 最大單筆獲利
        - 最大單筆虧損
    - 分為
        - 非多即空
        - 多
        - 空


#### 2.策略及定義假設
- **基本設定**
  - 標的：台指近月期貨指數
  - 資料頻率：日資料
  - 回測起始日：2000/01/01
  - 季線：60 日簡單移動平均線（SMA60）

- **交易規則**
  - 做多
    - 昨日收盤價 ≤ 昨日 SMA60
    - 今日收盤價 > 今日 SMA60
    - 視為**向上穿越季線**，建立多頭部位（Long）
  - 做空
    - 昨日收盤價 ≥ 昨日 SMA60
    - 今日收盤價 < 今日 SMA60
    - 視為**向下穿越季線**，建立空頭部位（Short）
  - 部位規則
    - 策略開始持有部位後，維持**非多即空**
    - 不存在空手狀態
    - 出現反向訊號時：
      1. 將原有部位平倉
      2. 同時建立反向部位

- **成交假設**
  - 訊號以**當日收盤價**判斷
  - 假設以**訊號當日收盤價成交**

- **損益定義**
  - 損益先以**加權指數點數**表示
  - 暫不考慮：
    - 初始本金
    - 每點價值
    - 交易成本
    - 手續費
    - 稅費
    - 滑價
      - 原本預期成交的價格，和實際真正成交的價格之間的差距。
      - 例如你看到加權指數在 20,000 點出現買進訊號，理論上希望用 20,000 點成交，但實際下單後可能成交在 20,005 點，這多出來的 5 點就是滑價。
      - 滑價常見原因包括市場快速波動、流動性不足、買賣價差，以及從訊號產生到訂單真正進市場之間的時間差。

- **交易定義**
  - 一筆完整交易定義為：
    - **進場 → 持有 → 出場**
  - 反手時：
    - 原部位完成一筆交易
    - 同時開始下一筆反向交易
  - 尚未出場的最後一筆部位視為**未平倉部位（Open Position）**

- **持有時間**
  - 持有時間以**交易日（Trading Days）**計算
  - 定義：
    - 今日收盤進場，下一個交易日收盤出場，持有時間為 1 個交易日
  - 不使用日曆日計算，因此週末及休市日不額外計入持有時間

- **績效指標**
  - 總交易
      - 完成「進場 → 出場」的交易筆數
  - 總損益
    - 所有已完成交易之損益點數加總
  - 勝率
    - 獲利交易次數占全部已完成交易次數的比例
  - 平均賺賠比
    - 平均獲利交易的獲利點數，相對於平均虧損交易虧損點數絕對值的比例
  - 平均持有期
    - 每筆交易平均持有多久
  - 最大回檔
    - 從歷史資產最高點開始，之後曾經最多跌掉多少。

  - 最大單筆獲利
  - 最大單筆虧損

- **第一筆部位如何建立**
  - 等待回測起始日後第一次穿越訊號再進場

##### 3.程式內容

載入資料

In [68]:
from pathlib import Path
import pandas as pd
file_path = Path("../../../input/ZTXA_20260902.xlsx")

df = pd.read_excel(
    file_path,
    header=1
)


清洗資料

In [69]:
df = df.rename(columns={
    "日期": "trade_date",
    "收盤價": "close_price"
})
df = df[["trade_date", "close_price"]]
df = df.sort_values("trade_date").reset_index(drop=True)
df["trade_date"] = pd.to_datetime(
    df["trade_date"],
    errors="coerce"
)

建立 sma

In [70]:
df["sma60"] = df["close_price"].rolling(60).mean()

建立訊號
- signal : Enum
    - "Long"
    - "Short"
    - "-"

In [71]:
def get_signal(index):
    currValue = df.loc[index, "close_price"]
    currSma = df.loc[index, "sma60"]

    yestValue = df.loc[index - 1, "close_price"]
    yestSma = df.loc[index - 1, "sma60"]

    if currValue > currSma and yestValue <= yestSma:
        # print(index, "L")
        return "Long"

    elif currValue < currSma and yestValue >= yestSma:
        # print(index, "S")
        return "Short"
    else:
        return "-"

df["signal"] = "-"
for index in  range(60, len(df)):
    df.loc[index, "signal"] = get_signal(index)

獨立訊號為表並取 2000-1-1 後

In [72]:
signal_df = df.loc[
    (df["signal"] != "-") & (df["trade_date"] >= pd.Timestamp("2000-01-01")),
    ["trade_date", "close_price", "sma60", "signal"]
].copy()
signal_df = signal_df.reset_index(drop=True)
signal_df.tail()

,trade_date,close_price,sma60,signal
379,2026-08-10,44987,44508.583333,Long
380,2026-08-19,44612,44977.116667,Short
381,2026-08-21,45148,44999.200000,Long
382,2026-08-24,44762,45014.533333,Short
383,2026-08-25,45027,45010.233333,Long


建立表格基礎數據

In [73]:
def get_profit(index):
    currValue = signal_df.loc[index, "close_price"]
    yestValue = signal_df.loc[index - 1, "close_price"]
    isLong = signal_df.loc[index -1, "signal"] == "Long"
    
    if(isLong):
        return currValue - yestValue
    else: 
        return yestValue - currValue
    
def get_hold_day(index):
    currDay = signal_df.loc[index, "trade_date"]
    yestDay = signal_df.loc[index - 1, "trade_date"]
    hold_day = (currDay - yestDay).days
    return hold_day

def get_profit_pct(index):

    value = signal_df.loc[index - 1, "close_price"]
    profit = signal_df.loc[index - 1, "profit"]
    return profit / value

def set_acc_profit(index):
    global signal_df
    
    if(index == 0):
        signal_df.loc[index, "all_acc_profit"] = 0
        signal_df.loc[index, "long_acc_profit"] = 0
        signal_df.loc[index, "short_acc_profit"] = 0
        return
    
    isLong = signal_df.loc[index, "signal"] == "Long"
    profit = signal_df.loc[index, "profit"]
    
    all_acc_profit = signal_df.loc[index - 1, "all_acc_profit"]
    long_acc_profit = signal_df.loc[index - 1, "long_acc_profit"]
    short_acc_profit = signal_df.loc[index - 1, "short_acc_profit"]
    
    all_acc_profit += profit
    if(isLong):
        long_acc_profit += profit
    else:
        short_acc_profit += profit
    
    signal_df.loc[index, "all_acc_profit"] = all_acc_profit
    signal_df.loc[index, "long_acc_profit"] = long_acc_profit
    signal_df.loc[index, "short_acc_profit"] = short_acc_profit

def set_hist_and_drawdown(index):
    global signal_df
    
    if(index == 0):
            signal_df.loc[index, "all_hist_max_acc"] = 0
            signal_df.loc[index, "long_hist_max_acc"] = 0
            signal_df.loc[index, "short_hist_max_acc"] = 0
            return
    
    all_hist_max_acc = signal_df.loc[index - 1, "all_hist_max_acc"]
    long_hist_max_acc = signal_df.loc[index - 1, "long_hist_max_acc"]
    short_hist_max_acc = signal_df.loc[index - 1, "short_hist_max_acc"]
    
    all_acc_profit = signal_df.loc[index, "all_acc_profit"]
    long_acc_profit = signal_df.loc[index, "long_acc_profit"]
    short_acc_profit = signal_df.loc[index, "short_acc_profit"]
    
    if(all_hist_max_acc < all_acc_profit):
        all_hist_max_acc = all_acc_profit
        
    if(long_hist_max_acc < long_acc_profit):
        long_hist_max_acc = long_acc_profit
        
    if(short_hist_max_acc < short_acc_profit):
        short_hist_max_acc = short_acc_profit
    
    all_drawdown = all_acc_profit - all_hist_max_acc
    long_drawdown = long_acc_profit - long_hist_max_acc
    short_drawdown = short_acc_profit - short_hist_max_acc
    
    signal_df.loc[index, "all_hist_max_acc"] = all_hist_max_acc
    signal_df.loc[index, "long_hist_max_acc"] = long_hist_max_acc
    signal_df.loc[index, "short_hist_max_acc"] = short_hist_max_acc
    
    signal_df.loc[index, "all_drawdown"] = all_drawdown
    signal_df.loc[index, "long_drawdown"] = long_drawdown
    signal_df.loc[index, "short_drawdown"] = short_drawdown



for index in  range(1, len(signal_df)):
    signal_df.loc[index - 1, "profit"] = get_profit(index)
    signal_df.loc[index - 1, "profit_pct"] = get_profit_pct(index)
    signal_df.loc[index - 1, "hold_day"] = get_hold_day(index)
    
    set_acc_profit(index - 1)
    set_hist_and_drawdown(index - 1)

signal_df.tail()

,trade_date,close_price,sma60,signal,profit,profit_pct,hold_day,all_acc_profit,long_acc_profit,short_acc_profit,all_hist_max_acc,long_hist_max_acc,short_hist_max_acc,all_drawdown,long_drawdown,short_drawdown
379,2026-08-10,44987,44508.583333,Long,-375.0,-0.008336,9.0,23216.0,29096.0,-5880.0,27265.0,30177.0,8832.0,-4049.0,-1081.0,-14712.0
380,2026-08-19,44612,44977.116667,Short,-536.0,-0.012015,2.0,22680.0,29096.0,-6416.0,27265.0,30177.0,8832.0,-4585.0,-1081.0,-15248.0
381,2026-08-21,45148,44999.200000,Long,-386.0,-0.008550,3.0,22294.0,28710.0,-6416.0,27265.0,30177.0,8832.0,-4971.0,-1467.0,-15248.0
382,2026-08-24,44762,45014.533333,Short,-265.0,-0.005920,1.0,22029.0,28710.0,-6681.0,27265.0,30177.0,8832.0,-5236.0,-1467.0,-15513.0
383,2026-08-25,45027,45010.233333,Long,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


 - 取得統計資訊
    - 總交易 trades
    - 淨損益 netProfit
    - 勝率 win
    - 平均賺賠比 avgWin/loss
    - 平均持有期 avgDuration
    - 最大回檔 MDD
    - 最大單筆獲利 MaxWin
    - 最大單筆虧損 MaxLoss

計算統計數據

In [74]:
all_trades = len(signal_df)
long_trades = len(signal_df[signal_df["signal"] == "Long"])
short_trades = len(signal_df[signal_df["signal"] == "Short"])

all_acc_profit = signal_df.loc[all_trades - 2, "all_acc_profit"]
long_acc_profit = signal_df.loc[all_trades - 2, "long_acc_profit"]
short_acc_profit = signal_df.loc[all_trades - 2, "short_acc_profit"]

all_win_count = len(signal_df["profit"] > 0)
all_loss_count = len(signal_df["profit"] < 0)
all_win_rate = all_win_count / (all_win_count + all_loss_count)

long_win_count = len(
    signal_df[
        (signal_df["profit"] > 0) & 
        (signal_df["signal"] =="Long")
        ]
    )
long_loss_count = len(
    signal_df[
        (signal_df["profit"] < 0) & 
        (signal_df["signal"] =="Long")
        ]
    )
long_win_rate = long_win_count / (long_win_count + long_loss_count)

short_win_count = len(
    signal_df[
        (signal_df["profit"] > 0) & 
        (signal_df["signal"] =="Short")
        ]
    )
short_loss_count = len(
    signal_df[
        (signal_df["profit"] < 0) & 
        (signal_df["signal"] =="Short")
        ]
    )
short_win_rate = short_win_count / (short_win_count + short_loss_count)


all_win_profit_mean = abs(signal_df.loc[signal_df["profit"] > 0, "profit"].mean())
all_loss_profit_mean = abs(signal_df.loc[signal_df["profit"] < 0, "profit"].mean())

all_profit_mean_ratio = all_win_profit_mean / all_loss_profit_mean

long_win_profit_mean = abs(signal_df.loc[
    (signal_df["profit"] > 0) & (signal_df["signal"] == "Long"),
    "profit"
].mean())
long_loss_profit_mean = abs(signal_df.loc[
    (signal_df["profit"] < 0) & (signal_df["signal"] == "Long"),
    "profit"
].mean())
long_profit_mean_ratio = long_win_profit_mean / long_loss_profit_mean 

short_win_profit_mean = abs(signal_df.loc[
    (signal_df["profit"] > 0) & (signal_df["signal"] == "Short"),
    "profit"
].mean())
short_loss_profit_mean = abs(signal_df.loc[
    (signal_df["profit"] < 0) & (signal_df["signal"] == "Short"),
    "profit"
].mean())
short_profit_mean_ratio = short_win_profit_mean / short_loss_profit_mean 

all_avg_duration = signal_df["hold_day"].mean()
long_avg_duration = signal_df.loc[signal_df["signal"] == "Long", "hold_day"].mean()
short_avg_duration = signal_df.loc[signal_df["signal"] == "Short", "hold_day"].mean()

all_max_drawdown = signal_df["all_drawdown"].min()
long_max_drawdown = signal_df["long_drawdown"].min()
short_max_drawdown = signal_df["short_drawdown"].min()

all_max_win = signal_df["profit"].max()
long_max_win = signal_df.loc[signal_df["signal"] == "Long","profit"].max()
short_max_win = signal_df.loc[signal_df["signal"] == "Short","profit"].max()

all_max_loss = signal_df["profit"].min()
long_max_loss = signal_df.loc[signal_df["signal"] == "Long","profit"].min()
short_max_loss = signal_df.loc[signal_df["signal"] == "Short","profit"].min()


In [75]:
def calc_stats(trades,profit, win_rate, profit_mean_ratio, avg_duration,mdd, max_win, max_loss):
    return {
        "交易次數": trades,
        "淨損益": profit,
        "勝率":win_rate,
        "平均賺賠比":profit_mean_ratio,
        "平均持有期":avg_duration,
        "最大回檔":mdd,
        "最大單筆獲利":max_win,
        "最大單筆虧損":max_loss,
    }

建立總結表格

In [76]:
all_stat = calc_stats(
    trades=all_trades,
    profit=all_acc_profit,
    win_rate=all_win_rate,
    profit_mean_ratio=all_profit_mean_ratio,
    avg_duration=all_avg_duration,
    mdd=all_max_drawdown,
    max_win=all_max_win,
    max_loss=all_max_loss
)

long_stat = calc_stats(
    trades=long_trades,
    profit=long_acc_profit,
    win_rate=long_win_rate,
    profit_mean_ratio=long_profit_mean_ratio,
    avg_duration=long_avg_duration,
    mdd=long_max_drawdown,
    max_win=long_max_win,
    max_loss=long_max_loss
)

short_stat = calc_stats(
    trades=short_trades,
    profit=short_acc_profit,
    win_rate=short_win_rate,
    profit_mean_ratio=short_profit_mean_ratio,
    avg_duration=short_avg_duration,
    mdd=short_max_drawdown,
    max_win=short_max_win,
    max_loss=short_max_loss
)


summary_df = pd.DataFrame(
    [all_stat, long_stat, short_stat],
    index=["非多即空","僅多單","僅空單"]
)
display(
    summary_df.style.format({
        "淨損益": "{:.0f}",
        "勝率": "{:.2%}",
        "平均賺賠比": "{:.2f}",
        "平均持有期": "{:.2f}天",
        "最大回檔": "{:.0f}",
        "最大單筆獲利": "{:.0f}",
        "最大單筆虧損": "{:.0f}"
    })
)

,交易次數,淨損益,勝率,平均賺賠比,平均持有期,最大回檔,最大單筆獲利,最大單筆虧損
非多即空,384,22029,50.00%,5.59,25.22天,-8287,9323,-1652
僅多單,192,28710,25.79%,6.80,31.13天,-4326,9323,-1120
僅空單,192,-6681,14.58%,4.43,19.35天,-15513,4081,-1652
